# keybed segmentation, full recipe

Runs `kvt.trainseg` as it runs locally: the procedural renderer supplies most of each batch and
the rendered corpus rides along. The renderer is what carries generalisation to real photos, so
training on the corpus alone collapses (measured: 481 px against 16 px).

Inputs: the `kvt-src` and `keybed-corpus` datasets. No camera frames leave the machine.

In [ ]:
import json
import shutil
import sys
from pathlib import Path

import torch

# the dataset carries the modules under whatever folder name it was zipped with, and the
# import has to be `kvt`, so the package is copied to that name on the writable disk
SRC = next(Path("/kaggle/input").rglob("trainseg.py")).parent
CORPUS = next(Path("/kaggle/input").rglob("corners.json")).parent
PACKAGE = Path("/kaggle/working/kvt")
SYNTH = Path("/kaggle/working/synth")
OUT = Path("/kaggle/working/keybed_seg.pt")

shutil.rmtree(PACKAGE, ignore_errors=True)
shutil.copytree(SRC, PACKAGE)
sys.path.insert(0, str(PACKAGE.parent))

print("src", SRC, "corpus", CORPUS)
print("cuda", torch.cuda.is_available(), torch.cuda.device_count(), "gpu(s)")

In [ ]:
# load_synth reads a png beside a sidecar, so the baked corpus is expanded back into that shape
corners = json.loads((CORPUS / "corners.json").read_text())
SYNTH.mkdir(parents=True, exist_ok=True)
for stem, quad in corners.items():
    shutil.copyfile(CORPUS / "frames" / f"{stem}.png", SYNTH / f"{stem}.png")
    (SYNTH / f"{stem}.json").write_text(
        json.dumps(
            {
                "kind": "synth",
                "startedAt": 0,
                "durationMs": 0,
                "corners": [{"x": x, "y": y} for x, y in quad],
                "imageWidth": 288,
                "imageHeight": 288,
                "mimeType": "image/png",
            }
        )
    )
print(f"{len(corners)} frames staged at {SYNTH}")

In [ ]:
import multiprocessing

from kvt.trainseg import train_seg

model, iou = train_seg(
    train_samples=20_000,
    val_samples=800,
    epochs=12,
    workers=multiprocessing.cpu_count(),
    real_fraction=0.0,
    synth_dir=SYNTH,
    synth_fraction=0.4,
)
torch.save(model.state_dict(), OUT)
print(f"saved {OUT} val_iou {iou:.4f}")